# vast.ai — run the v15 three-pass agent on ls20 (RTX PRO 6000)

Purpose: observe the full scout → TTT → plan loop on ONE game (ls20, a
holdout the prior never trained on) with real GPU budgets, and produce a
time-accounting so we can size the wall-clock governor before Kaggle.

Run cells top to bottom. Everything uses the dedicated venv at
`/workspace/venv` (the proven v14 recipe — Jupyter's kernel pip and the
shell pip are different interpreters; never trust bare `python`).

In [ ]:
# Cell 1 — clone or update the repo
GIT_TOKEN = ""          # fill in if the repo is private
BRANCH = "main"
import os, subprocess
url = f"https://{GIT_TOKEN + '@' if GIT_TOKEN else ''}github.com/shreyasmahimkar/arc-agi-3.git"
if os.path.exists('/workspace/arc3/.git'):
    print(subprocess.run(['git', '-C', '/workspace/arc3', 'pull'],
                         capture_output=True, text=True).stdout)
else:
    print(subprocess.run(['git', 'clone', '-b', BRANCH, url, '/workspace/arc3'],
                         capture_output=True, text=True).stdout)
!ls /workspace/arc3/CommunitySolutions/chronos_solver/v15/

In [ ]:
# Cell 2 — dedicated venv (idempotent; ~2 min first time)
PY = "/workspace/venv/bin/python"
import os
if not os.path.exists(PY):
    !python3 -m venv /workspace/venv --system-site-packages
    !{PY} -m pip install --ignore-installed blinker
    !{PY} -m pip install \
        /workspace/arc3/arc-prize-2026-arc-agi-3/arc_agi_3_wheels/arcengine-0.9.3-py3-none-any.whl \
        /workspace/arc3/arc-prize-2026-arc-agi-3/arc_agi_3_wheels/arc_agi-0.9.8-py3-none-any.whl \
        python-dotenv scipy
!{PY} -c "import torch, scipy, arc_agi, arcengine; print('venv OK | torch', torch.__version__, '| cuda', torch.cuda.is_available())"

In [ ]:
# Cell 3 — GPU + module contracts
PY = "/workspace/venv/bin/python"
!{PY} -c "import torch; print(torch.cuda.get_device_name(0))"
!cd /workspace/arc3/CommunitySolutions/chronos_solver/v15 && {PY} -m plm.smoke

In [ ]:
# Cell 4 — prior weights
# Preferred: drag your Mac-trained plm_weights.pt into the Jupyter file
# browser, then move it into place (uploads land in /workspace):
#   !mv /workspace/plm_weights.pt /workspace/arc3/CommunitySolutions/chronos_solver/v15/
# Fallback below: if no weights exist, train a quick 4-game pilot prior
# on this box (~25 min) — reuses v14's tokenizer if available.
PY = "/workspace/venv/bin/python"
import os
V15 = '/workspace/arc3/CommunitySolutions/chronos_solver/v15'
if not os.path.exists(f'{V15}/plm_weights.pt'):
    !cp /workspace/arc3/CommunitySolutions/chronos_solver/v14/plm_weights.pt {V15}/ 2>/dev/null || true
if not os.path.exists(f'{V15}/plm_weights.pt'):
    print('no weights anywhere - training pilot prior now')
    !cd {V15} && {PY} gen_data.py --games ar25,bp35,cn04,dc22 \
        --episodes-per-game 150 --max-steps 100 --expert-frac 0.5 \
        --out /workspace/v15_pilot_shards
    !cd {V15} && {PY} train_wm.py --phase all --shards /workspace/v15_pilot_shards \
        --epochs 5 --steps-per-epoch 500 --bsz 256
!{PY} -c "import torch; s=torch.load('/workspace/arc3/CommunitySolutions/chronos_solver/v15/plm_weights.pt', map_location='cpu', weights_only=True); print('weights keys:', list(s))"

In [ ]:
# Cell 5 — THE RUN: ls20, blind (holdout game; prior never saw it).
# Two modes — pick by editing BFS budget:
#   V15_BFS_BUDGET=0    -> simulates Kaggle exactly (no engine shortcut):
#                          scout -> TTT -> plan only
#   V15_BFS_BUDGET=600  -> full local tier: real BFS first, expert episode
#                          seeds the TTT, bfs-exec on solved levels
import subprocess, time
PY = "/workspace/venv/bin/python"
subprocess.run("pkill -f play_game.py || true", shell=True); time.sleep(2)
cmd = (
    "cd /workspace/arc3/CommunitySolutions/chronos_solver/v15 && "
    "nohup env V15_BFS_BUDGET=600 V15_REQUIRE_WEIGHTS=1 "
    "V15_SCOUT_ACTIONS=80 V15_TTT_SECONDS=180 V15_THINK_BUDGET=120 "
    "V15_RESCOUT_ACTIONS=40 V15_STUCK_WINDOW=60 "
    f"{PY} play_game.py --game ls20 --fast --max-steps 300 "
    "> /workspace/v15_ls20_runner.log 2>&1 &"
)
subprocess.Popen(cmd, shell=True); time.sleep(10)
!tail -20 /workspace/v15_ls20_runner.log

In [ ]:
# Cell 6 — monitor (re-run me any time)
!ps aux | grep play_game | grep -v grep || echo '*** RUN FINISHED (or crashed - check log) ***'
!echo --- && nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader
!echo --- && tail -25 /workspace/v15_ls20_runner.log | grep -E 'Reasoning|TTT|pass1|Advanced|scout|finished' || tail -10 /workspace/v15_ls20_runner.log

In [ ]:
# Cell 7 — TIME ACCOUNTING: who spent the wall clock?
# This is the measurement that sizes the Kaggle wall-clock governor.
import re
from datetime import datetime
from collections import Counter

LOG = '/workspace/v15_ls20_runner.log'
lines = open(LOG, errors='ignore').read().splitlines()
ts_re = re.compile(r'^(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2},\d{3})')

def ts(line):
    m = ts_re.match(line)
    return datetime.strptime(m.group(1), '%Y-%m-%d %H:%M:%S,%f') if m else None

tags = Counter()
tag_time = Counter()
prev_t, prev_tag = None, None
ttt_s = 0.0
for ln in lines:
    t = ts(ln)
    if t is None:
        continue
    if 'TTT done' in ln:
        m = re.search(r"'seconds': ([\d.]+)", ln)
        if m:
            ttt_s += float(m.group(1))
    m = re.search(r'Reasoning: (\w+[:\-]?\w*)', ln)
    if m:
        tag = m.group(1).split('(')[0]
        tags[tag] += 1
        if prev_t is not None:
            tag_time[tag] += (t - prev_t).total_seconds()
        prev_t = t

total = (ts([l for l in lines if ts(l)][-1]) - ts([l for l in lines if ts(l)][0])).total_seconds()
print(f'total wall clock : {total/60:.1f} min')
print(f'TTT (training)   : {ttt_s/60:.1f} min')
print('\nactions by reasoning tag:')
for k, v in tags.most_common():
    print(f'  {k:30s} {v:5d} actions  ~{tag_time[k]/60:6.1f} min between-step time')
print('\n-> the tag owning the most minutes is what the governor must cap')

## Reading the run

- `pass1-BFS: budget out before L0` (BFS_BUDGET=0) — Kaggle simulation confirmed
- `scout(N left)` ×80 → `V15 TTT #1` → `TTT done {...}` — passes 1→2
- `plm:bfs-soft(p=...)` — value-guided commits; watch whether p is nonzero on a game the prior NEVER saw (this is the zero-shot + TTT question)
- `plm:think(Np)-...` — deep-think escalations; **count these in cell 7** — they're the suspected 9-hour killer on Kaggle
- `V15: no progress in 60 actions — re-scouting` → `TTT #2` — the stuck loop
- scorecard at the end: levels_completed and per-level actions vs ls20 baselines [22,123,73,84,96,192,186]

When done: download `/workspace/v15_ls20_runner.log`, then DESTROY the instance.